# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [1]:
!pip install -q schedule pytest # установка библиотек, если ещё не
print('ok')

ok


ЋвЄ § ­® ў ¤®бвгЇҐ.


In [2]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import requests
import schedule
from bs4 import BeautifulSoup
import re
from datetime import datetime
print('ok')

ok


## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [3]:
 def get_book_data(book_url: str) -> dict:
    """
    Этот код парсит информацию с сайта книг.
    На вход подаем html ссылку в формает строки.
    book_name возвращает название книги в str, используя поиск по тэгу h1;
    book_price возвращает цену книги В ФУНТАХ в float;
    book_rank возвращает ранг книги - кол-во звездочек;
    book_count возвращает кол-во книг в наличии в int, используя поиск по нужному классу;
    book_info возвращает описание книги в str;
    cleaned_book_add возвращает в иде словаря информацию из таблицы КАК ОНА ЕСТЬ.
    """
    
    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    response = requests.get(book_url)  # используем get-запрос
    soup = BeautifulSoup(response.content, 'html.parser')

    if response.status_code == 200:
        book_name = soup.find('h1').text.strip()
        book_price = float(soup.find(class_='price_color').text.strip()[1:])
        book_rank = soup.find(class_='star-rating')['class'][1]
        
        book_count_text = soup.find(class_='instock availability').text.strip()
        book_count = int(''.join(re.findall(r'[1-9]', book_count_text)))

        book_info_element = soup.select_one('#product_description + p')
        book_info = book_info_element.text.strip() if book_info_element else 'No description'

        table = soup.select_one('.table.table-striped')
        book_add = [row.text.strip() for row in table.find_all('tr')] if table else []
        cleaned_book_add = {}
        for item in book_add:
            for key in ['UPC', 'Product Type', 'Price (excl.tax)', 'Price (incl. tax)', 'Tax', 'Availability', 'Number of reviews']:
                if item.startswith(key):
                    value = item[len(key):].strip()  # создаем значение, которое начинается там, где заканчивается ключ
                    cleaned_book_add[key] = value
                    break
        result = {
            'book_name': book_name,
            'book_price': book_price,
            'book_rank': book_rank,
            'book_count': book_count,
            'book_info': book_info,
            'cleaned_book_add': cleaned_book_add
        }
    return result
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [4]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'book_name': 'A Light in the Attic',
 'book_price': 51.77,
 'book_rank': 'Three',
 'book_count': 22,
 'book_info': "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And 

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [5]:
def scrape_books(is_save: bool=False) -> list:
    """
    Парсит все 50 страниц из католога.
    Если is_save = True, то сохраняем информацию о книгах в виде books_data.txt.
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    res = []

    for page in range(1, 51):
        url = f'http://books.toscrape.com/catalogue/page-{page}.html'
        print(f'Страница {page}')

        soup = BeautifulSoup(requests.get(url).content, 'html.parser')

        for book in soup.select('article.product_pod'):
            book_link = book.select_one('h3 a')['href']
            book_url = f'http://books.toscrape.com/catalogue/{book_link.replace("../../../", "")}'

            book_data = get_book_data(book_url)
            res.append(book_data)

    if is_save:
        with open('books_data.txt', 'w', encoding='utf-8') as file:
            for book in res:
                file.write(f'{book}\n')
    return res
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ 

In [6]:
# Проверка работоспособности функции
res = scrape_books(is_save=True) # Допишите ваши аргументы
print(type(res), len(res)) # и проверки

Страница 1
Страница 2
Страница 3
Страница 4
Страница 5
Страница 6
Страница 7
Страница 8
Страница 9
Страница 10
Страница 11
Страница 12
Страница 13
Страница 14
Страница 15
Страница 16
Страница 17
Страница 18
Страница 19
Страница 20
Страница 21
Страница 22
Страница 23
Страница 24
Страница 25
Страница 26
Страница 27
Страница 28
Страница 29
Страница 30
Страница 31
Страница 32
Страница 33
Страница 34
Страница 35
Страница 36
Страница 37
Страница 38
Страница 39
Страница 40
Страница 41
Страница 42
Страница 43
Страница 44
Страница 45
Страница 46
Страница 47
Страница 48
Страница 49
Страница 50
<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [ ]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ
def schedule_parsing_19():
    print('Запускаем автоматический парсинг!')

    books_data = scrape_books(is_save=True)

    print('Парсинг завершен!')

schedule.every().day.at('14:06').do(schedule_parsing_19)

while True:
    schedule.run_pending()
    time.sleep(60)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [ ]:
import os

if not os.path.exists('tests'):
    os.makedirs('tests')

with open('tests/test_scraper.py', 'w') as f:
    f.write('''import sys
import os

sys.path.append('C:\\\\Users\\\\cgv\\\\anaconda3')

from main import get_book_data, scrape_books

def test_get_book_data_returns_dict():
    test_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
    result = get_book_data(test_url)
    assert type(result) == dict

def test_get_book_data_has_book_name():
    test_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
    result = get_book_data(test_url)
    assert 'book_name' in result
    assert result['book_name'] != ''

def test_scrape_books_returns_list():
    result = scrape_books(is_save=False)
    assert type(result) == list''')

print('Файл с тестами создан!')

In [ ]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest tests/test_scraper.py
print('Ошибок нет')

## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```

In [ ]:
import os

project_name = "books_scraper"
os.makedirs(project_name, exist_ok=True)
os.chdir(project_name)

folders = ['artifacts', 'notebooks', 'tests']
for folder in folders:
    os.makedirs(folder, exist_ok=True)

files = ['scraper.py', 'README.md', 'requirements.txt', '.gitignore']
for file in files:
    open(file, 'w').close()

print('Структура проекта создана!')